# Suite BRE — The POSIX BRE dialect

`Expressions.POSIX.BRE` models POSIX's basic regular expressions (IEEE Std 1003.1, XBD 9.3, its grammar 9.5.2), as grep
reads its patterns, a newline-separated list of them, and as C++'s `std::regex` reads them under its `basic` and `grep`
grammars. Its kinds are validated as its grammar has them, bind nothing, cover POSIX.1's four editions, print as a BRE
writes them, are refused by evaluation by name, and are checked complete against XBD 9.5 kept as data.

In [ ]:
import json
from pathlib import Path

from mbse.Expressions import Framework
from mbse.Expressions.POSIX import BRE, ERE
from mbse.Expressions.POSIX.BRE import Expressions as B, Text as BText
from mbse.Expressions.POSIX.ERE import Expressions as R
from mbse.Expressions.Framework import Evaluators as FE, Standards as FS, Symbolics as SY, Terms as F
from mbse.Schemas.Framework import Builders, JSON, Validators
from support import raises

store = Builders.Store()  # the POSIX family's dialects, ERE's install first: the installs run in any order
for install in [Framework.install, ERE.install, BRE.install]:
    store.transaction(install)
store.transaction(BRE.define)  # its editions


def term(spec, dialect=B.DIALECT):
    """The term value `spec` writes, the root of `dialect`'s holder, written in a transaction."""
    with store.transaction():
        family, own = dialect.path.split(".")
        return F.term(getattr(getattr(store.Expressions, family), own).Expression(lambda e: e.root(dialect.spec(spec))))


validate = lambda spec: B.DIALECT.validate(term(spec))  # noqa: E731
text = lambda spec: BText.ToText(term(spec))  # noqa: E731
union = lambda name: [b.name for b in store.registered(name).branches]  # noqa: E731: a union's branches
char = lambda c, spelling=None: lambda u: u.character(  # noqa: E731: a character, of its code point, and its spelling
    lambda r: r.value(ord(c)) if spelling is None else r.raw(spelling).value(ord(c)))
alt = lambda *elements: lambda u: u.alternative(lambda r: r.elements(list(elements)))  # noqa: E731
pattern = lambda *alternatives: lambda u: u.pattern(lambda r: r.alternatives(list(alternatives)))  # noqa: E731
capture = lambda *alternatives: lambda u: u.capturinggroup(lambda r: r.alternatives(list(alternatives)))  # noqa: E731
def repeat(element, least, most=None, spelling=None):
    """A quantifier: `element\\{least,most\\}`, as `spelling` writes it where given."""
    def write(r):
        r = r.min(least).element(element)
        r = r if most is None else r.max(most)
        return r if spelling is None else r.raw(spelling)
    return lambda u: u.quantifier(write)


cls = lambda *elements, negate=False: lambda u: u.characterclass(lambda r: r.negate(negate).elements(list(elements)))  # noqa: E731
span = lambda low, high: lambda u: u.characterclassrange(lambda r: r.min(low).max(high))  # noqa: E731
chars = lambda kind: lambda u: u.characterset(lambda r: r.setKind(kind))  # noqa: E731
dot = chars("any")
anchor = lambda kind: lambda u: u.assertion(lambda r: r.assertionKind(kind))  # noqa: E731
backref = lambda n: lambda u: u.backreference(lambda r: r.ref(n))  # noqa: E731
named = lambda kind, name: lambda u: getattr(u, kind)(lambda r: r.name(name))  # noqa: E731: `[:alpha:]`, `[.a.]`, `[=a=]`
patterns = lambda *specs: lambda u: u.patternlist(lambda r: r.patterns(list(specs)))  # noqa: E731
error = lambda source: lambda u: u.error(lambda r: r.text(source))  # noqa: E731
missing = lambda kind: lambda u: u.missing(lambda r: r.kind(kind))  # noqa: E731
abc = pattern(alt(char("a"), char("b")), alt(char("c")))  # `ab\|c`

<a id="BRE-01"></a>
## BRE-01 · Its union and holder: its kinds and invalid syntax, a branch of the framework's terms, reading its family's other dialect; a holder refining the framework's, round-tripped through JSON

In [ ]:
assert B.TAGS == ("patternlist", "pattern", "alternative", "capturinggroup", "quantifier", "characterclass",
                  "characterclassrange", "characterset", "character", "assertion", "backreference", "exclass",
                  "collatingelement", "equivalence") and B.HELPERS == () and B.SINCE == {}
assert list(B.DIALECT.kinds()) == union(B.DIALECT.union) == [*B.TAGS, "error", "missing"]
assert B.DIALECT.name() == "POSIX BRE" and B.DIALECT.tag == "posix_bre" and B.DIALECT.union == "Expressions.POSIX.BRE.Term"
assert "posix_bre" in union("Expressions.Framework.Term") and "posix_bre" in union("Expressions.Framework.Dialect")
assert B.DIALECT.reads() == (R.DIALECT,) and R.DIALECT.reads() == (B.DIALECT,)  # its family's other dialect
Holder = store.registered("Expressions.POSIX.BRE.Expression")
assert Holder.ref and Holder.refines is store.registered("Expressions.Framework.Expression") and F.is_expression(Holder)
assert Holder.properties["root"].schema is store.registered(B.DIALECT.union) and list(Holder.adjacencies) == ["abi"]
with store.transaction():
    holder = store.Expressions.POSIX.BRE.Expression(lambda e: e.root(B.DIALECT.spec(abc)))
assert Validators.Validate(store)(Holder, holder) == [] and B.DIALECT.validate(holder) == []
assert F.dialect_of(F.term(holder)) == B.DIALECT.union and F.kind_of(F.term(holder))[0] == "pattern"
assert F.same(JSON.FromJSON(store)(Holder, JSON.ToJSON(store)(Holder, holder)).root, holder.root)
with raises(Exception, match="'group' is not a member of a union value"):  # its root holds its own terms: a BRE has no group
    with store.transaction():
        store.Expressions.POSIX.BRE.Expression(lambda e: e.root(B.DIALECT.spec(lambda u: u.group(lambda r: r))))
assert B.DIALECT.validate(term(repeat(repeat(char("a"), 1), 0), R.DIALECT)) == []  # an ERE's term, as POSIX ERE reads it

<a id="BRE-02"></a>
## BRE-02 · Its kinds: their properties in order, each term property its Term; forms and make round-trip

In [ ]:
props = lambda name: list(store.registered(f"Expressions.POSIX.BRE.{name}").properties)  # noqa: E731
assert props("PatternList") == ["patterns"] and props("Pattern") == props("CapturingGroup") == ["alternatives"]
assert props("Alternative") == ["elements"] and props("Quantifier") == ["raw", "min", "max", "element"]  # never lazy
assert props("CharacterClass") == ["negate", "elements"] and props("CharacterClassRange") == ["min", "max"]
assert props("CharacterSet") == ["setKind"] and props("Character") == ["raw", "value"] and props("Assertion") == ["assertionKind"]
assert props("Backreference") == ["ref"] and props("ExClass") == props("CollatingElement") == props("Equivalence") == ["name"]
prop = lambda name, member: store.registered(f"Expressions.POSIX.BRE.{name}").properties[member].schema  # noqa: E731
Term = store.registered(B.DIALECT.union)
assert prop("Quantifier", "element") is prop("CharacterClassRange", "min") is prop("CharacterClassRange", "max") is Term
assert prop("PatternList", "patterns").element is prop("Pattern", "alternatives").element is prop("Alternative", "elements").element is Term
made = F.form(term(pattern(alt(repeat(char("a"), 2, 3), backref(1)))))
assert made.kind == "pattern" and [F.kind_of(a)[0] for a in made.arguments] == ["alternative"]
quantified = F.form(term(repeat(cls(span(char("a"), char("z"))), 1)))
assert quantified.attributes == {"min": 1} and [F.kind_of(a)[0] for a in quantified.arguments] == ["characterclass"]
for spec in [patterns(abc, pattern(alt(dot, anchor("end")))), capture(alt(backref(1))),
             cls(named("exclass", "alpha"), named("collatingelement", "ch"), named("equivalence", "e"), char("-"), negate=True)]:
    assert F.same(term(B.DIALECT.make(F.form(term(spec)))), term(spec))

<a id="BRE-03"></a>
## BRE-03 · Its checks: what each kind's terms may be, as XBD 9.5.2 has them, where an anchor stands, and its attributes' values

In [ ]:
for spec, problems in [
    (patterns(alt(char("a"))), ["a patternlist's patterns cannot hold an alternative"]),
    (pattern(char("a")), ["a pattern's alternatives cannot hold a character"]),
    (alt(pattern(alt())), ["an alternative's elements cannot hold a pattern"]),
    (alt(named("exclass", "alpha")), ["an alternative's elements cannot hold an exclass"]),
    (capture(char("a")), ["a capturinggroup's alternatives cannot hold a character"]),
    (repeat(repeat(char("a"), 1), 0), ["a quantifier's element cannot hold a quantifier"]),  # nondupl_BRE BRE_dupl_symbol
    (repeat(anchor("start"), 0), ["a quantifier's element cannot hold an assertion"]),  # `^*`: its `*` a character
    (cls(alt()), ["a characterclass's elements cannot hold an alternative"]),
    (cls(dot), ["a characterclass's elements cannot hold a characterset"]),  # `.` within brackets is a character
    (span(dot, named("collatingelement", "z")), ["a characterclassrange's min cannot hold a characterset"]),
    (repeat(char("a"), -1), ["a quantifier's min cannot be negative, got -1"]),
    (repeat(char("a"), 3, 2), ["a quantifier's max cannot be less than its min, got 2 and 3"]),
    (lambda u: u.quantifier(lambda r: r.element(char("a"))), ["a quantifier needs a min"]),
    (lambda u: u.quantifier(lambda r: r.min(1)), ["a quantifier needs an element"]),
    (chars("digit"), ["CharacterSet.setKind cannot be 'digit'"]),
    (lambda u: u.characterset(lambda r: r), ["a characterset needs a setKind"]),
    (lambda u: u.character(lambda r: r.value(0x110000)), ["a character's value must be a code point, got 1114112"]),
    (lambda u: u.character(lambda r: r.raw("a")), ["a character needs a value"]),
    (anchor("word"), ["Assertion.assertionKind cannot be 'word'"]),
    (lambda u: u.assertion(lambda r: r), ["an assertion needs an assertionKind"]),
    (backref(0), ["a backreference's ref must be 1 to 9, got 0"]),
    (backref(10), ["a backreference's ref must be 1 to 9, got 10"]),  # BACKREF: a single digit
    (named("exclass", "a:b"), ["an exclass's name cannot hold ':'"]),
    (named("collatingelement", "a.="), ["a collatingelement's name cannot hold '.'"]),
    (named("equivalence", ""), ["an equivalence needs a name"]),
    (alt(char("a"), anchor("start")), ["a start assertion stands only where a BRE begins"]),  # L_ANCHOR
    (alt(anchor("end"), char("a")), ["an end assertion stands only where a BRE ends"]),  # R_ANCHOR
    (capture(alt(anchor("start"), char("a"))), ["a BRE's subexpression holds no anchor"]),  # 9.3.8: the implementation's
]:
    assert validate(spec) == problems, (validate(spec), problems)
for spec in [abc, alt(), pattern(alt(anchor("start"), char("a"), anchor("end")), alt(anchor("start"))),
             cls(span(char("a"), named("collatingelement", "z")), char("-")), repeat(backref(9), 0), capture(alt())]:
    assert validate(spec) == [], spec

<a id="BRE-04"></a>
## BRE-04 · grep's list of patterns: each a BRE of its own, anchored where it begins and ends, printed a line each

In [ ]:
listed = patterns(pattern(alt(anchor("start"), char("a"))), pattern(alt(char("b"), anchor("end"))), abc)
assert validate(listed) == [] and text(listed) == "^a\nb$\nab\\|c"  # grep's pattern_list (XCU grep)
assert validate(patterns(pattern(alt(char("a"), anchor("start"))))) == [
    "argument 0: argument 0: a start assertion stands only where a BRE begins"]
assert [F.kind_of(t)[0] for t in F.walk(term(patterns(pattern(alt(char("a"))))))] == [
    "patternlist", "pattern", "alternative", "character"]

<a id="BRE-05"></a>
## BRE-05 · What it binds: nothing; a subexpression numbered, a backreference resolving it by its number

In [ ]:
assert not any(F.is_binder(b.schema) for b in store.registered(B.DIALECT.union).branches)
grouped = term(pattern(alt(capture(alt(char("a"))), backref(1))))
assert SY.free(grouped, B.DIALECT) == set() and B.DIALECT.validate(grouped) == []
assert [F.kind_of(t)[0] for t in F.walk(grouped)] == ["pattern", "alternative", "capturinggroup", "alternative",
                                                      "character", "backreference"]

<a id="BRE-06"></a>
## BRE-06 · Invalid syntax: error and missing in its Term, validated, printed as written, refused by name

In [ ]:
assert validate(error("\\(")) == ["invalid syntax '\\\\('"] and text(error("\\(")) == "\\("
assert validate(missing("atom")) == ["missing atom"] and text(alt(char("a"), missing("atom"))) == "a"
assert validate(pattern(alt(char("a"), error("[")))) == ["argument 0: argument 1: invalid syntax '['"]
assert text(pattern(alt(char("a"), error("[z")))) == "a[z"
with raises(ValueError, match="POSIX BRE error has no text to print"):
    text(lambda u: u.error(lambda r: r))

<a id="BRE-07"></a>
## BRE-07 · Its editions: POSIX.1's four, each by its designation and common names, the ISO/IEC designations C++ cites among them, each with every kind

In [ ]:
bre_ = store.Instances.Expressions.POSIX.BRE
assert [FS.tags(s) for s in FS.standards(bre_).values()] == [
    ("IEEE Std 1003.1-2001", "POSIX.1-2001", "ISO/IEC 9945:2003"),
    ("IEEE Std 1003.1-2008", "POSIX.1-2008", "ISO/IEC/IEEE 9945:2009"),
    ("IEEE Std 1003.1-2017", "POSIX.1-2017"), ("IEEE Std 1003.1-2024", "POSIX.1-2024")]
assert B.EDITIONS[1] == ("IEEE Std 1003.1-2008", 2008, ("POSIX.1-2008", "ISO/IEC/IEEE 9945:2009"), "POSIX_2008")
assert FS.find(bre_, "ISO/IEC/IEEE 9945:2009") is FS.find(bre_, "POSIX.1-2008")  # the designation C++23 cites
edition = FS.properties(FS.find(bre_, "POSIX.1-2024"))
assert edition["bodies"] == ("org.ieee",) and edition["number"] == "1003.1" and edition["revision"] == ("year", 2024)
assert all([s.name for s in FS.kinds(e)] == [f"Expressions.POSIX.BRE.{name}" for name in (
    "PatternList", "Pattern", "Alternative", "CapturingGroup", "Quantifier", "CharacterClass", "CharacterClassRange",
    "CharacterSet", "Character", "Assertion", "Backreference", "ExClass", "CollatingElement", "Equivalence")]
           for e in FS.standards(bre_).values())
assert store.transaction(BRE.define)[0]["POSIX_2008"] is FS.find(bre_, "POSIX.1-2008")  # run again, it writes nothing

<a id="BRE-08"></a>
## BRE-08 · Printing: every kind as a BRE writes it, its special characters escaped, a spelling as written; an ERE's term as an ERE

In [ ]:
every = pattern(alt(anchor("start"), capture(alt(char("a")), alt(char("."))), repeat(char("b"), 0), repeat(char("c"), 1),
                    repeat(char("d"), 0, 1), repeat(char("e"), 2, 2), repeat(char("f"), 2), repeat(char("g"), 2, 5),
                    cls(span(char("a"), char("z")), named("exclass", "digit"), named("collatingelement", "ch"),
                        named("equivalence", "e"), char("]"), negate=True), dot, backref(1), anchor("end")))
assert text(every) == "^\\(a\\|\\.\\)b*c\\{1,\\}d\\{0,1\\}e\\{2\\}f\\{2,\\}g\\{2,5\\}[^a-z[:digit:][.ch.][=e=]]].\\1$"
assert text(abc) == "ab\\|c"  # POSIX.1-2024's alternation
assert text(pattern(alt(char("+"), char("*"), char("?"), char("$"), char("\n"), cls(char("\\"), char("*"))))) == "+\\*?\\$\n[\\*]"
assert text(pattern(alt(repeat(char("a"), 1, spelling="\\+"), char("x", "\\x")))) == "a\\+\\x"  # as written
assert BText.ToText(term(pattern(alt(repeat(char("a"), 0, 1), capture(alt(char("b"))))), R.DIALECT)) == "a?(b)"  # an ERE's

<a id="BRE-09"></a>
## BRE-09 · Evaluation: each kind refused by its dialect's name

In [ ]:
interpreter = FE.Interpreter(B.DIALECT, {}, evaluated=frozenset())  # no kind evaluated, each refused by its dialect's name
for spec in [patterns(abc), abc, char("a"), cls(), backref(1)]:
    with raises(NotImplementedError, match=f"POSIX BRE {F.kind_of(term(spec))[0]} is not evaluated"):
        interpreter(term(spec))
with raises(ValueError, match="POSIX BRE error is not evaluated"):
    interpreter(term(error("\\(")))

<a id="BRE-10"></a>
## BRE-10 · Completeness: grammars/posix-regex.json maps every production and token of XBD 9.5, POSIX.1-2024's too, and every kind is mapped

In [ ]:
grammar = json.loads(Path("../../grammars/posix-regex.json").read_text(encoding="utf-8"))
nodes = grammar["nodes"]
assert len(nodes) == 57 and len({n["node"] for n in nodes}) == 57 and "57 productions and tokens" in grammar["checked"]
assert all(set(n) - {"note"} in ({"node", "source", "kinds"}, {"node", "source", "not"}) for n in nodes)
mapped = {kind for n in nodes for kind in n.get("kinds", ())}
assert {kind for kind in mapped if kind.startswith("BRE.")} == {f"BRE.{tag}" for tag in B.TAGS}  # every kind, exactly
by = {n["node"]: n for n in nodes}
renamed = {"RE_expression": "BRE_branch", "simple_RE": "simple_BRE", "nondupl_RE": "nondupl_BRE",
           "one_char_or_coll_elem_RE": "one_char_or_coll_elem_BRE", "RE_dupl_symbol": "BRE_dupl_symbol"}
assert all(by[old].get("kinds") == by[new].get("kinds") for old, new in renamed.items())  # POSIX.1-2024's names, alike
assert all("not" in by[token] for token in ("Back_qm", "Back_plus", "Back_bar"))  # its `\?`, `\+`, `\|`: a raw, a printing
assert by["pattern_list"]["kinds"] == ["BRE.patternlist", "ERE.patternlist"]  # grep's, XCU grep